Import all the necessary library. Read the patient diabetic record file. Read the header of the file.

In [ ]:
import pandas as pd
import numpy as np


In [ ]:

logistic_record=pd.read_csv(r"diabetes_record_logistic.csv")
print(logistic_record.head())

Replace the "zero" values of the column like age, glucose etc that is not possible in actual life witn NaN so the numpy recognize them and then replace them with the median of their own column 

In [ ]:
zero_columns=["Glucose","BloodPressure","SkinThickness","BMI","Age"]
logistic_record[zero_columns]=logistic_record[zero_columns].replace(0,np.nan)
# logistic_record=logistic_record.dropna()
for column in zero_columns:
    logistic_record[column] = (
        logistic_record[column]
        .fillna(logistic_record[column].median())
    )
logistic_record.isnull().sum()



Set the value of the features (logistic_X) and the output(logistic_Y)

In [ ]:
logistic_X=logistic_record.drop(["Outcome"],axis=1)
logistic_y=logistic_record["Outcome"]



Feature Scaling

In [ ]:
logistic_mean=logistic_X.mean(axis=0)
logistic_std=logistic_X.std(axis=0)
normalization_l=(logistic_X-logistic_mean)/(logistic_std)
print(normalization_l)

Sigmoid Function

In [ ]:
def sigmoid(z):
    z= np.clip(z, -500, 500)
    return 1/(1+np.exp(-z))

Compute the Cost

In [ ]:
def calculate_cost(X, y, w, b):

    m = X.shape[0]

    z = np.dot(X, w) + b
    prediction = sigmoid(z)

    eps = 1e-15
    prediction = np.clip(prediction, eps, 1-eps)

    cost = (
        -y * np.log(prediction)
        -(1-y) * np.log(1-prediction)
    )

    return np.sum(cost) / m

Compute Gradient

In [ ]:
def compute_gradient_l(x,y,w,b,sigmoid_fun):
    m=x.shape[0]
    z=(np.dot(x,w)+b)
    prediction=sigmoid_fun(z)
    subtraction=prediction-y
    w=np.dot(x.T,subtraction)/m
    b=np.sum(subtraction)/m
    return w,b

Compute Graient Descent

In [ ]:
def gradient_descent_l(x,y,w,b,alpha,gradient_fun,sigmoid_fun,num_iter_l,cost_function):
    for i in range(num_iter_l):
        compute_cost=cost_function(x,y,w,b)
        dj_dw, dj_db = gradient_fun(x, y, w, b,sigmoid_fun)   
        w = w - alpha * dj_dw               
        b = b - alpha * dj_db 
        if i % 1000 == 0:
                    print(f"Iteration {i}: Cost={compute_cost:.2f}")
    return w,b

Prediction compare the actual and the target values

In [ ]:
num_iter_l=10000
logistic_X = normalization_l.values
logistic_y=logistic_y.values
alpha_l=0.9
logistic_b=0
m=logistic_X.shape[0]
logistic_w=np.array([0,0,0,0,0,0,0,0])
w,b=gradient_descent_l(logistic_X,logistic_y,logistic_w,logistic_b,alpha_l,compute_gradient_l,sigmoid,num_iter_l,calculate_cost) 
prediction=(np.dot(logistic_X,w)+b) 
sig_prediction=sigmoid(prediction)
predictions = (sig_prediction >= 0.5).astype(int)

logistic_y = logistic_y.flatten()

for i in range(m):
    print(f"Patient {i+1} -> Probability: {sig_prediction[i]:.2f} | Predicted Class: {predictions[i]} | Actual Target: {logistic_y[i]}")



Find accuracy of the model

In [ ]:
# 6. FIX: Use 'predictions' (integers) to evaluate true model accuracy
accuracy = np.mean(predictions == logistic_y) * 100
print(f"\nFinal Model Accuracy: {accuracy:.2f}%")

User Input. user enter the neceessary information and model predict whether the patient has the diabetes or not.

In [ ]:

print("        Diabetes Prediction System        ")

Pregnancies = float(input("Pregnancies: "))
Glucose = float(input("Glucose: "))
BloodPressure = float(input("Blood Pressure: "))
SkinThickness = float(input("Skin Thickness: "))
Insulin = float(input("Insulin: "))
BMI = float(input("BMI: "))

DiabetesPedigreeFunction = float(
    input("Diabetes Pedigree Function: ")
)

Age = float(input("Age: "))

new_patient = pd.DataFrame({
    "Pregnancies": [Pregnancies],
    "Glucose": [Glucose],
    "BloodPressure": [BloodPressure],
    "SkinThickness": [SkinThickness],
    "Insulin": [Insulin],
    "BMI": [BMI],
    "DiabetesPedigreeFunction": [
        DiabetesPedigreeFunction
    ],
    "Age": [Age]
})

# Normalize using training mean/std
new_patient_normalized = (
    new_patient - logistic_mean
) / logistic_std

# Convert to NumPy
new_patient_record = (
    new_patient_normalized.values.flatten()
)

# Calculate probability
probability = sigmoid(
    np.dot(w, new_patient_record) + b
)

# Apply threshold
prediction = int(probability >= 0.5)

print("Prediction Results")

print(
    f"Diabetes Probability: "
    f"{probability * 100:.2f}%"
)

if prediction == 1:
    print("Prediction: Diabetes")
else:
    print("Prediction: No Diabetes")